In [1]:
import os
import time
import numpy as np
import struct
import xml.etree.ElementTree as ET
from pynq import Overlay, MMIO, allocate

# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------
BITFILE   = "WNNAcceleratorBlk.bit" 
DATA_FILE = "mnist_fpga_test_data.npz"
LUT_DIR   = "luts" 

# Hardware Constants
NUM_LUTS    = 500
N_CLASSES   = 10
ADDR_BITS   = 6
M           = 1 << ADDR_BITS # 64

# Weight Packing Constants
LUT_DATA_WIDTH  = 8     
WORDS_PER_ENTRY = 4     

# Input Constants
INPUT_BITS       = 25088 
DMA_TRANSFER_LEN = INPUT_BITS // 32 # 784 words

# Register Map
REG_CTRL       = 0x00
REG_OUTPUT     = 0x04
CTRL_START_BIT = 0 
CTRL_DONE_BIT  = 1 
CTRL_CLEAR_BIT = 2 

# ---------------------------------------------------------------
# 1. SETUP & DRIVER SEARCH
# ---------------------------------------------------------------
print("Loading Overlay...")
if not os.path.exists(BITFILE): raise FileNotFoundError(f"Missing {BITFILE}")

overlay = Overlay(BITFILE) 

# 1.1 Find DMA and WNN IP
try:
    dma = overlay.axi_dma_0
    wnn_ip = overlay.wnn_axi_0 
except AttributeError:
    print("Standard names not found, searching ip_dict...")
    for name, ip in overlay.ip_dict.items():
        if 'wnn' in name.lower(): wnn_ip = MMIO(ip['phys_addr'], ip['addr_range'])
        if 'dma' in name.lower(): dma = getattr(overlay, name)

if not wnn_ip or not dma:
    raise Exception("Could not find WNN or DMA IP!")

# 1.2 Find BRAM Controller 
def get_bram_driver(overlay, bitfile_path):
    # Try finding by name first
    keys = [k for k in overlay.ip_dict.keys() if 'bram' in k.lower()]
    if keys: return overlay.ip_dict[keys[0]], overlay.ip_dict[keys[0]].mmio.length
    
    # Fallback: Parse HWH file
    hwh_path = bitfile_path.replace(".bit", ".hwh")
    if not os.path.exists(hwh_path): return None, 0
    tree = ET.parse(hwh_path)
    for module in tree.getroot().iter('MODULE'):
        if 'axi_bram_ctrl' in module.get('VLNV', '').lower():
            base = int(next(p.get('VALUE') for p in module.iter('PARAMETER') if p.get('NAME') == 'C_S_AXI_BASEADDR'), 16)
            high = int(next(p.get('VALUE') for p in module.iter('PARAMETER') if p.get('NAME') == 'C_S_AXI_HIGHADDR'), 16)
            return MMIO(base, high-base+1), high-base+1
    raise Exception("BRAM Controller not found!")

bram_ip, bram_size = get_bram_driver(overlay, BITFILE)
print("   -> Drivers Loaded (WNN, DMA, BRAM).")

# ---------------------------------------------------------------
# 2. PROGRAM BRAM
# ---------------------------------------------------------------
print(f"[STEP 2] Programming Weights into BRAM...")
start_prog = time.time()
MAX_VAL = (1 << LUT_DATA_WIDTH) - 1 

# Verify LUT dir exists
if not os.path.exists(LUT_DIR):
    raise FileNotFoundError(f"LUT Directory '{LUT_DIR}' not found. Upload 'luts' folder!")

for l in range(NUM_LUTS):
    mem_path = os.path.join(LUT_DIR, f"lut_{l:03d}.mem")
    if not os.path.exists(mem_path):
        raise FileNotFoundError(f"Missing {mem_path}")
        
    with open(mem_path, 'r') as f:
        lines = f.readlines()
        
    for addr_idx, line in enumerate(lines):
        if addr_idx >= M: break
        
        parts = line.strip().split()
        counts = [int(x, 16) for x in parts] 
        
        # Pack 10 classes into 128 bits
        packed_val = 0
        for c in range(N_CLASSES):
            val = counts[c]
            if val > MAX_VAL: val = MAX_VAL
            packed_val |= (val << (c * LUT_DATA_WIDTH))
            
        # Write to BRAM
        entry_offset = (l * M + addr_idx) * (WORDS_PER_ENTRY * 4)
        for w in range(WORDS_PER_ENTRY):
            chunk = (packed_val >> (w * 32)) & 0xFFFFFFFF
            bram_ip.write(entry_offset + (w*4), chunk)

print(f"   -> Weights Loaded in {time.time() - start_prog:.2f}s")

# ---------------------------------------------------------------
# 3. DATA PREPARATION
# ---------------------------------------------------------------
print("[STEP 3] Loading and Pre-Packing Data...")
data = np.load(DATA_FILE)
x_test_bits = data['x'] 
y_test = data['y']      
total_images = len(y_test)

# Allocate CMA Buffer
input_buffer = allocate(shape=(DMA_TRANSFER_LEN,), dtype=np.uint32)

# Pre-Pack
t_pack = time.time()
packed_dataset = np.zeros((total_images, DMA_TRANSFER_LEN), dtype=np.uint32)
for i in range(total_images):
    # Standard Little Endian Packing
    packed_bytes = np.packbits(x_test_bits[i].astype(np.uint8), bitorder='little')
    packed_dataset[i] = np.frombuffer(packed_bytes, dtype=np.uint32)
print(f"   -> Packing complete in {time.time() - t_pack:.2f}s")

# ---------------------------------------------------------------
# 4. FAST INFERENCE FUNCTION
# ---------------------------------------------------------------
def run_stream_inference(packed_image_row):
    # 1. Copy to CMA
    np.copyto(input_buffer, packed_image_row)
    
    # 2. DMA Transfer
    dma.sendchannel.transfer(input_buffer)
    dma.sendchannel.wait()
    
    # 3. Start Accelerator
    wnn_ip.write(REG_CTRL, (1 << CTRL_START_BIT)) # Pulse 1
    wnn_ip.write(REG_CTRL, 0)                     # Pulse 0
    
    # 4. Wait for Done
    while (wnn_ip.read(REG_CTRL) & (1 << CTRL_DONE_BIT)) == 0:
        pass
        
    # 5. Read Result
    pred = wnn_ip.read(REG_OUTPUT)
    
    # 6. Clear Status
    wnn_ip.write(REG_CTRL, (1 << CTRL_CLEAR_BIT))
    return pred

# ---------------------------------------------------------------
# 5. BENCHMARK
# ---------------------------------------------------------------
print(f"\n[STEP 4] Starting Benchmark ({total_images} images)...")

correct_count = 0
start_time = time.time()

# Sanity Check First
s_pred = run_stream_inference(packed_dataset[0])
print(f"   [Sanity] Img 0 -> Pred: {s_pred}, Actual: {y_test[0]}")
if s_pred != y_test[0]:
    print("   [WARNING] Sanity check failed. Check bit packing order or BRAM data.")

# Run All
for i in range(total_images):
    if run_stream_inference(packed_dataset[i]) == y_test[i]:
        correct_count += 1
        
    if (i + 1) % 1000 == 0:
        print(f"   -> {i + 1}/{total_images} | Acc: {correct_count/(i+1)*100:.2f}% | FPS: {(i+1)/(time.time()-start_time):.1f}")

total_time = time.time() - start_time
print("\n==================================================================")
print(f"FINAL RESULTS")
print(f"Accuracy:   {correct_count/total_images*100:.2f}%")
print(f"FPS:        {total_images/total_time:.2f}")
print("==================================================================")

# Cleanup
try:
    if hasattr(input_buffer, 'freebuffer'): input_buffer.freebuffer()
    del input_buffer
except: pass

     HYBRID TEST: OLD WEIGHT LOADER + NEW FAST STREAMER           
[STEP 1] Loading Overlay...


   -> Drivers Loaded (WNN, DMA, BRAM).
[STEP 2] Programming Weights into BRAM...
   -> Weights Loaded in 7.75s
[STEP 3] Loading and Pre-Packing Data...
   -> Packing complete in 8.02s

[STEP 4] Starting Benchmark (10000 images)...
   [Sanity] Img 0 -> Pred: 7, Actual: 7
   -> 1000/10000 | Acc: 94.80% | FPS: 1536.7
   -> 2000/10000 | Acc: 93.80% | FPS: 1476.0
   -> 3000/10000 | Acc: 94.10% | FPS: 1510.5
   -> 4000/10000 | Acc: 94.33% | FPS: 1522.3
   -> 5000/10000 | Acc: 94.36% | FPS: 1526.8
   -> 6000/10000 | Acc: 94.80% | FPS: 1531.2
   -> 7000/10000 | Acc: 95.00% | FPS: 1534.1
   -> 8000/10000 | Acc: 95.35% | FPS: 1535.3
   -> 9000/10000 | Acc: 95.68% | FPS: 1537.8
   -> 10000/10000 | Acc: 95.64% | FPS: 1538.9

FINAL RESULTS
Accuracy:   95.64%
FPS:        1538.71
